In [1]:
from py_sas_studio_custom_steps import CustomStep

In [2]:
from duckdb_assistant import DuckDBAssistant

In [3]:
from dotenv import load_dotenv

In [4]:
dda = DuckDBAssistant()

['gemini-3.8-flash', 'gemini-3.7-flash', 'gemini-3.6-flash', 'gemini-3.5-flash']


In [6]:
csv_path = "https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv"

In [7]:
prompt = f"""

For a CSV table residing at {csv_path}, I want to ingest this into a DuckDB table and also have a path to save out
rejected records to a rejects table. Give me SQL for the same.

"""

In [8]:
import os
dda.generate(prompt,sql_file_path=os.path.join(os.getcwd(),"reject.sql"))

{'query': "INSTALL httpfs;\nLOAD httpfs;\n\nCREATE OR REPLACE TABLE hmeq AS\nSELECT *\nFROM read_csv(\n    'https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv',\n    store_rejects = true,\n    rejects_table = 'hmeq_rejects',\n    rejects_scan = 'hmeq_reject_scans'\n);",
 'response_text': "```sql\nINSTALL httpfs;\nLOAD httpfs;\n\nCREATE OR REPLACE TABLE hmeq AS\nSELECT *\nFROM read_csv(\n    'https://raw.githubusercontent.com/sassoftware/sas-viya-dmml-pipelines/refs/heads/master/data/hmeq.csv',\n    store_rejects = true,\n    rejects_table = 'hmeq_rejects',\n    rejects_scan = 'hmeq_reject_scans'\n);\n```"}

In [11]:
dda.sql(sql_file_path=os.path.join(os.getcwd(),"reject.sql"))

In [13]:
dda.dd.sql("select * from hmeq_reject_scans")

scan_idUBIGINT,file_idUBIGINT,file_pathVARCHAR,delimiterVARCHAR,quoteVARCHAR,escapeVARCHAR,newline_delimiterVARCHAR,skip_rowsUINTEGER,has_headerBOOLEAN,columnsVARCHAR,date_formatVARCHAR,timestamp_formatVARCHAR,user_argumentsVARCHAR


In [14]:
cs = CustomStep(os.path.join(os.getcwd(),"..","DuckDB - CSV to Parquet.step"))

In [15]:
prompt = """
Revamp this custom step program to also incorporate an option to check if users want to store rejects and if true, incorporate
code to save rejects and reject scans to specified table names, also making the tables permanent.
"""

In [16]:
cs.create_sas_program(prompt)

Error with model gemini-3.1-pro-preview: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro, description: Generate content input tokens per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-3.1-pro, description: Generate content input tokens per minute limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests

'Program generated and attached to the custom step object successfully.'

In [18]:
cs.extract_sas_program()

'/************************************************************************\nDuckDB - CSV to Parquet Conversion with Error Rejection Handling\n\nPurpose: This SAS program utilizes SAS/ACCESS to DuckDB to ingest a CSV file,\n         convert it into a Parquet file, and optionally handle rejects.\nInput Parameters:\n- csv_file_path: Full path to the source CSV file (macro variable)\n- has_headers: Boolean (TRUE/FALSE) indicating if the CSV has a header row (macro variable)\n- parquet_output_path: Folder location for the output Parquet file (macro variable)\n- parquet_file_name: Name/prefix for the output Parquet file (macro variable)\n- save_rejects: Boolean (TRUE/FALSE) to enable rejection handling (macro variable)\n- reject_data_table: Name of the permanent table to store rejected rows (macro variable)\n- reject_scan_table: Name of the permanent table to store reject scanning metadata (macro variable)\n\nCreated: Sundaresh Sankaran\nVersion: 1.1.0 (21OCT2026)\n**************************

In [20]:
dda.generate(explain_results = True, prompt = "does duckdb sql accept 1 and 0 for True and false?")

<module 'google.genai.errors' from '/Users/sinsrn/current_projects/DuckDB-csv-to-parquet/build/.venv/lib/python3.14/site-packages/google/genai/errors.py'>
Error with model gemini-3.8-flash: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Retrying with next model after 2.53 seconds...


Yes, **DuckDB supports using `1` and `0` for `TRUE` and `FALSE`, but with important distinctions between explicit casting and implicit usage**:

---

### 1. Explicit Casting
DuckDB allows converting integers to booleans using explicit casts (`::BOOLEAN` or `CAST(... AS BOOLEAN)`):
* `0` converts to `FALSE`.
* Any non-zero integer (such as `1`, `-1`, etc.) converts to `TRUE`.

```sql
SELECT 
    1::BOOLEAN AS true_val,    -- returns TRUE
    0::BOOLEAN AS false_val,   -- returns FALSE
    CAST(42 AS BOOLEAN) AS non_zero_val; -- returns TRUE
```

---

### 2. Comparisons (`=` operator)
DuckDB allows direct equality checks between integers and boolean values:
```sql
SELECT 1 = TRUE;   -- returns TRUE
SELECT 0 = FALSE;  -- returns TRUE
SELECT 1 = FALSE;  -- returns FALSE
```

---

### 3. Logical Contexts (`WHERE`, `HAVING`, `CASE WHEN`, `AND` / `OR`)
Unlike SQLite or MySQL (which treat integers as booleans natively), DuckDB is strongly typed. In standard filtering and logical expressions, DuckDB expects an actual boolean expression:

* **Fails / Disallowed implicitly:**
  ```sql
  -- May produce a Binder Error: WHERE clause must evaluate to a boolean, not INTEGER
  SELECT * FROM my_table WHERE 1;
  ```
* **Correct DuckDB usage:**
  ```sql
  SELECT * FROM my_table WHERE 1::BOOLEAN;
  -- or simply
  SELECT * FROM my_table WHERE TRUE;
  ```

---

### Summary
* **Literal conversion / Storage:** You can insert or cast `1` and `0` into `BOOLEAN` columns (`1::BOOL`, `0::BOOL`).
* **Conditionals:** Always use explicit boolean literals (`TRUE`, `FALSE`) or cast integer flags (`col::BOOLEAN`) in conditional statements.

{'query': 'SELECT \n    1::BOOLEAN AS true_val,    -- returns TRUE\n    0::BOOLEAN AS false_val,   -- returns FALSE\n    CAST(42 AS BOOLEAN) AS non_zero_val; -- returns TRUE',
 'response_text': 'Yes, **DuckDB supports using `1` and `0` for `TRUE` and `FALSE`, but with important distinctions between explicit casting and implicit usage**:\n\n---\n\n### 1. Explicit Casting\nDuckDB allows converting integers to booleans using explicit casts (`::BOOLEAN` or `CAST(... AS BOOLEAN)`):\n* `0` converts to `FALSE`.\n* Any non-zero integer (such as `1`, `-1`, etc.) converts to `TRUE`.\n\n```sql\nSELECT \n    1::BOOLEAN AS true_val,    -- returns TRUE\n    0::BOOLEAN AS false_val,   -- returns FALSE\n    CAST(42 AS BOOLEAN) AS non_zero_val; -- returns TRUE\n```\n\n---\n\n### 2. Comparisons (`=` operator)\nDuckDB allows direct equality checks between integers and boolean values:\n```sql\nSELECT 1 = TRUE;   -- returns TRUE\nSELECT 0 = FALSE;  -- returns TRUE\nSELECT 1 = FALSE;  -- returns FALSE\n```\

In [21]:
dda.generate(explain_results = True, prompt = "What is the significance of union_by_name in read_csv?")

In DuckDB, the `union_by_name` parameter in `read_csv` (or `read_csv_auto`) controls how multiple CSV files are combined when their column layouts do not match identically.

By default (`union_by_name = false`), DuckDB expects all files matching a glob pattern or file list to have identical schemas in the exact same column order as the first file inspected. If files differ in schema or column ordering, reading them by default can lead to runtime errors, misaligned data, or unexpected nulls.

Setting `union_by_name = true` resolves this by matching and merging columns based on their header names rather than their positional order:

### Key Features and Significance

1. **Column Order Independence**:
   * If `file1.csv` has columns `(id, name, amount)` and `file2.csv` has `(amount, id, name)`, DuckDB aligns each field under the correct column name instead of mapping values positionally.

2. **Handling Schema Evolution (Missing and Extra Columns)**:
   * If newer files have added columns (e.g., `(id, name, email)`) while older files lack them (e.g., `(id, name)`), DuckDB produces a unified table containing the superset of all columns `(id, name, email)`. Rows from files lacking a column are automatically populated with `NULL`.

3. **Type Promotion Across Files**:
   * If the inferred type for a column differs across files (for instance, integer values in one file and decimals or strings in another), DuckDB automatically resolves the column to the lowest common compatible supertype.

4. **Eliminates Manual ETL / Pre-processing**:
   * Without `union_by_name`, reading hundreds of heterogeneous files requires writing custom union scripts or standardizing file schemas upfront. `union_by_name = true` allows querying across varying CSV files in a single pass.

{'query': '',
 'response_text': 'In DuckDB, the `union_by_name` parameter in `read_csv` (or `read_csv_auto`) controls how multiple CSV files are combined when their column layouts do not match identically.\n\nBy default (`union_by_name = false`), DuckDB expects all files matching a glob pattern or file list to have identical schemas in the exact same column order as the first file inspected. If files differ in schema or column ordering, reading them by default can lead to runtime errors, misaligned data, or unexpected nulls.\n\nSetting `union_by_name = true` resolves this by matching and merging columns based on their header names rather than their positional order:\n\n### Key Features and Significance\n\n1. **Column Order Independence**:\n   * If `file1.csv` has columns `(id, name, amount)` and `file2.csv` has `(amount, id, name)`, DuckDB aligns each field under the correct column name instead of mapping values positionally.\n\n2. **Handling Schema Evolution (Missing and Extra Columns

In [22]:
dda.generate(explain_results = True, prompt = "If I have a store_rejects table of abc which I guess is temporary how do I immediately save it for future reference?")

```sql
-- Option 1: Immediately persist the table to a standalone Parquet file (Recommended)
COPY store_rejects TO 'store_rejects.parquet' (FORMAT PARQUET);

-- Option 2: Alternatively, persist directly to a new or existing DuckDB database file
ATTACH 'my_saved_data.db' AS persistent_db;
CREATE TABLE persistent_db.store_rejects AS SELECT * FROM store_rejects;
```

### Explanation

* **`COPY ... TO 'store_rejects.parquet' (FORMAT PARQUET);`**: This is the fastest and most portable way in DuckDB to snapshot a table or temporary result. It writes the entire contents of `store_rejects` to a compressed, columnar Parquet file on your disk. You can query this file anytime in the future directly without loading it first (e.g., `SELECT * FROM 'store_rejects.parquet'`).
* **`ATTACH ... AS persistent_db` & `CREATE TABLE ... AS SELECT`**: If you want to keep the data inside a persistent DuckDB database catalog rather than a standalone file, `ATTACH` mounts a database file (creating it if it does not exist) and copies the table contents over from your current temporary/in-memory session.

{'query': "-- Option 1: Immediately persist the table to a standalone Parquet file (Recommended)\nCOPY store_rejects TO 'store_rejects.parquet' (FORMAT PARQUET);\n\n-- Option 2: Alternatively, persist directly to a new or existing DuckDB database file\nATTACH 'my_saved_data.db' AS persistent_db;\nCREATE TABLE persistent_db.store_rejects AS SELECT * FROM store_rejects;",
 'response_text': "```sql\n-- Option 1: Immediately persist the table to a standalone Parquet file (Recommended)\nCOPY store_rejects TO 'store_rejects.parquet' (FORMAT PARQUET);\n\n-- Option 2: Alternatively, persist directly to a new or existing DuckDB database file\nATTACH 'my_saved_data.db' AS persistent_db;\nCREATE TABLE persistent_db.store_rejects AS SELECT * FROM store_rejects;\n```\n\n### Explanation\n\n* **`COPY ... TO 'store_rejects.parquet' (FORMAT PARQUET);`**: This is the fastest and most portable way in DuckDB to snapshot a table or temporary result. It writes the entire contents of `store_rejects` to a com

In [ ]:
dda.generate(explain_results = True, prompt = "What are the default names of the rejects and rejects_scan tables from read_csv?")

In [5]:
import os

In [7]:
custom_step = CustomStep(os.path.join(os.getcwd(),"..","CSV to Parquet.step"))

In [8]:
with open(os.path.join(os.getcwd(),"..","extras","CSV to Parquet.sas"),"w",encoding="utf-8") as sas_program:
    sas_program.write(custom_step.extract_sas_program())

with open(os.path.join(os.getcwd(),"..","extras","CSV to Parquet.json"),"w",encoding="utf-8") as ui:
    ui.write(custom_step.extract_ui())

